# Проверка REST API Loginom и GigaChat
Практическая работа №4. Заполните `.env` в корне проекта. Не вставляйте секретные ключи в ноутбук и не коммитьте их.

In [ ]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd

load_dotenv('../.env')
LOGINOM_BASE_URL = os.getenv('LOGINOM_BASE_URL', 'https://edu.loginom.dev/lgi/rest').rstrip('/')
LOGINOM_PACKAGE = os.environ['LOGINOM_PACKAGE']
USER_ID = 1  # замените на ID существующего пользователя
payload = {'Variables': {'user_id': USER_ID}}
CHECK_URL = f"{LOGINOM_BASE_URL}/{LOGINOM_PACKAGE}/CheckUserId"
HISTORY_URL = f"{LOGINOM_BASE_URL}/{LOGINOM_PACKAGE}/GetUserHistory"


## 1. Проверка существования пользователя
Ожидаемый контракт `CheckUserId`: `DataSet.Rows[0].user_exists` равен 1 или 0. Сверьте фактический ответ с `/lgi/rest/help`.

In [ ]:
check_response = requests.post(CHECK_URL, json=payload, timeout=45)
check_response.raise_for_status()
check_data = check_response.json()
check_data


In [ ]:
check_rows = check_data.get('DataSet', {}).get('Rows', [])
if not check_rows:
    raise RuntimeError('CheckUserId не вернул DataSet.Rows')
user_exists = str(check_rows[0].get('user_exists', '0')).strip().casefold() in {'1','true','yes','да'}
if not user_exists:
    raise ValueError(f'Пользователь {USER_ID} не найден')
print(f'Пользователь {USER_ID} найден')


## 2. Получение истории покупок

In [ ]:
history_response = requests.post(HISTORY_URL, json=payload, timeout=45)
history_response.raise_for_status()
history_data = history_response.json()
rows = history_data['DataSet']['Rows']
df = pd.DataFrame(rows)
display(df.head(10))


## 3. Формирование контекста для GigaChat

In [ ]:
lines = []
for i, row in enumerate(rows[:10], start=1):
    lines.append(f"{i}. {row.get('product_name', 'Неизвестный товар')} — отдел: {row.get('department_rus', row.get('department', 'не указан'))}, категория: {row.get('aisle', 'не указана')}, покупок: {row.get('order_count', 0)}")
products_text = '\n'.join(lines)
print(products_text)


## 4. Запрос к GigaChat

In [ ]:
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage
api_key = os.getenv('GIGA_KEY')
if not api_key:
    raise ValueError('Ключ GIGA_KEY не найден в .env')
llm = GigaChat(credentials=api_key, model=os.getenv('GIGA_MODEL', 'GigaChat-2'), verify_ssl_certs=False, temperature=0.3, max_tokens=500)
system_prompt = 'Ты — персональный помощник покупателя. Анализируй только переданную историю, не выдумывай покупки и факты. Обращайся на вы. Дай краткий анализ и 1–2 совета, не более 5–6 предложений.'
user_prompt = f'Клиент user_id={USER_ID}. История покупок:\\n{products_text}'
answer = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
print(answer.content)


## 5. Контроль
Проверьте, что `CheckUserId` отклоняет несуществующий ID, `GetUserHistory` возвращает не более 10 строк, а товары относятся к выбранному клиенту. Убедитесь, что рекомендация не добавляет покупки, которых нет в истории. Тесты этого ноутбука требуют опубликованных сервисов и действующего ключа GigaChat.